In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json
os.chdir("..")

In [3]:
import torch
import numpy as np
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from tqdm.auto import tqdm
from datasets import load_dataset
from cluster_intrep_repo.utils import initialize_tokenizer, tokenize_blocksworld_generation, THINK_TOKEN, THINK_START_TOKEN



os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

compute_dtype = torch.bfloat16
device   = 'cuda'
model_id = "Qwen/QwQ-32B"

In [4]:
from pathlib import Path

cur_dir = Path(".").absolute()


def load_dataset_from_file(domain_name, task_name):
    prompt_dir = cur_dir / Path(f"./cot-planning/results/{domain_name}/qwq-32b/")
    with open(prompt_dir / f"{task_name}.json", 'r') as file:
        return json.load(file)

In [5]:
task_name = "plan_generation_po"
eval_results = [
    load_dataset_from_file(domain_name, task_name)["instances"] for domain_name in [
        "blocksworld_mystery",
        "blocksworld_mystery_2"
    ]
]
eval_results = [{x["dataset_idx"]: x for x in er} for er in eval_results]

In [6]:
tokenizer = initialize_tokenizer(model_id)

In [7]:
import re

def find_label_positions(label_dataset, dataset):
    label_positions = {}
    for idx in tqdm(label_dataset):
        instance = dataset[idx]
        label = label_dataset[idx]["label"]
        if label is None:
            continue
        label_positions[idx] = {}
        
        for tag, content in label_dataset[idx]["label"].items():
            if content is None:
                continue
            gen_ind = instance["generation"].find(content[:30])
            
            if gen_ind == -1:
                continue
            
            tokens = tokenize_blocksworld_generation(tokenizer, instance, instance["generation"][:gen_ind])
            
            if len(tokens[0]) < 5000:
                continue
            
            label_positions[idx][tag] = len(tokens[0])
        
        label_positions[idx]["total"] = len(tokenize_blocksworld_generation(tokenizer, instance)[0])
            
    return label_positions

In [8]:
gen_datasets = [
    load_dataset(f"dmitriihook/qwq-32b-planning-{x}")["train"] for 
    x in ["mystery-24k", "mystery-2-24k"]
]

In [9]:
label_datasets = [
    load_dataset(f"dmitriihook/blocksworld-mystery-qwq-reasoning-parts-exploration")["train"],
    load_dataset(f"dmitriihook/blocksworld-mystery-2-qwq-reasoning-parts-exploration")["train"],
]

In [10]:
label_datasets = [
    {x["index"]: x for x in ld} for ld in label_datasets
]

In [11]:
label_postions_datasets = [
    find_label_positions(ld, dataset) for ld, dataset in zip(label_datasets, gen_datasets)
]

  0%|          | 0/400 [00:00<?, ?it/s]

  0%|          | 0/400 [00:00<?, ?it/s]

In [12]:
phrases = [
    [
    "attack",
    "succumb",
    "overcome",
    "feast",
    "province",
    "planet",
    "harmony",
    "craves",
    "pain"
    ],
    [
    "illuminate",
    "silence",
    "distill",
    "divest",
    ]
]

In [13]:
def extract_all_phrase_positions(tokens: torch.Tensor, phrase: str, cot_only: bool = True) -> list[str]:
    """Find end of the phrase token positions"""
    tokens = tokens.squeeze()

    phrase_tokens = [
        tokenizer.encode(" " + phrase),
        tokenizer.encode(" " + phrase.capitalize()),
        tokenizer.encode("\n" + phrase)[1:],
        tokenizer.encode("\n" + phrase.capitalize())[1:],
        tokenizer.encode("\n\n" + phrase)[1:],
        tokenizer.encode("\n\n" + phrase.capitalize())[1:],
    ]

    positions = set()

    if cot_only:
        start_pos = torch.where(tokens == 151667)[0]
        start_mask = torch.arange(tokens.shape[0]) >= start_pos

    for phts in phrase_tokens:
        presence_mask = torch.ones_like(tokens)
        if cot_only:
            presence_mask = presence_mask * start_mask

        for i, t in enumerate(phts):
            presence_mask = presence_mask * (tokens == t)[i:]
            presence_mask = presence_mask[:-1]

        for p in (torch.where(presence_mask)[0]).tolist():
            positions.add(
                tuple([p-1, p + len(phts)])
            )        
    
    return sorted(list(set(positions)))

In [14]:
model     = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=compute_dtype, attn_implementation="sdpa", 
                                                device_map="auto")

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Loading checkpoint shards:   0%|          | 0/14 [00:00<?, ?it/s]

In [15]:
from collections import defaultdict

def collect_hidden_states(ids: list[int], dataset, layers: list[int]) -> dict[int, dict]:
    hidden_states = defaultdict(dict)
    for idx in tqdm(ids):
        row = dataset[idx]
        tokens = tokenize_blocksworld_generation(tokenizer, row)
        with torch.no_grad():
            hs = model(tokens.to(device), output_hidden_states=True).hidden_states
            for layer in layers:
                hidden_states[idx][layer] = hs[layer][0].cpu().to(torch.float16).numpy()

    return hidden_states

In [16]:
n_rows = 40
layer = 47

In [17]:
def collect_correct_ids(eval_results: dict) -> list[int]:
    clean_ids = []
    for idx in range(303):
        if eval_results[idx]["llm_correct"]:
            clean_ids.append(idx)
        if len(clean_ids) == n_rows:
            break

    return clean_ids

correct_ids =[
    collect_correct_ids(eval_results[0]),
    collect_correct_ids(eval_results[1])
]

In [18]:
collected_hidden_states = [
    collect_hidden_states(correct_ids[i], gen_datasets[i], [layer]) for i in range(2)
] 

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

In [19]:
def collect_mean_representations(ids: list[int], dataset, positions_dataset, hidden_states, layer: int, phrases: list[str], n_pos: int=3, offset: int=0) -> np.ndarray:
    reprs = defaultdict(list)
    for idx in tqdm(ids):
        if idx not in positions_dataset:
            continue
        if "final-plan-formulation-verification" not in positions_dataset[idx]:
            continue
        end_pos = positions_dataset[idx]["final-plan-formulation-verification"]
        row = dataset[idx]
        hs: torch.Tensor = hidden_states[idx][layer]

        generation = row["generation"]
        text = generation.split("</think>")[0]

        tokens = tokenize_blocksworld_generation(tokenizer, row, text)[0]

        phrase_positions = [
            extract_all_phrase_positions(tokens, phrase) for phrase in phrases
        ]

        if any(len(x) < n_pos + offset + 2 for x in phrase_positions):
            continue

        for phrase, positions in zip(phrases, phrase_positions):
            positions = [p for p in positions if p[1] < end_pos]
            # positions = extract_all_phrase_positions(tokens, phrase)
            # print(len(positions))
            _reprs = []
            for ps, pe in positions[-n_pos - offset:-offset]:
                _reprs.append(hs[ps:pe].mean(axis=0))
            reprs[phrase].append(np.stack(_reprs, axis=0).mean(axis=0))

    return {k: np.stack(v, axis=0) for k, v in reprs.items()}

In [20]:
offset = 10

mean_reprs = {
    i: collect_mean_representations(correct_ids[i], gen_datasets[i], label_postions_datasets[i], collected_hidden_states[i], layer, phrases[i], n_pos=5, offset=offset) for i in range(2)
}

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

In [21]:
mean_reprs_clean = {k: np.stack(v).mean(0) for k, v in mean_reprs[0].items()}
mean_reprs_mystery = {k: np.stack(v).mean(0) for k, v in mean_reprs[1].items()}

mean_clean = np.stack([mean_reprs_clean[p] for p in phrases[0]], axis=0).mean(0)
mean_mystery = np.stack([mean_reprs_mystery[p] for p in phrases[1]], axis=0).mean(0)

In [88]:

idx = 7
row = gen_datasets[0][idx]

text = "\n\n".join(row["generation"].split("\n\n")[:40])

tokens = tokenize_blocksworld_generation(tokenizer, row, text)[:, :-2]

action_postions = {
   phrase: extract_all_phrase_positions(tokens, phrase, cot_only=False)
    for phrase in phrases[0]
}

In [89]:
phrase = "attack"

print(
    tokenizer.encode(" " + phrase),
    tokenizer.encode(" " + phrase.capitalize()),
    tokenizer.encode("\n" + phrase.capitalize()),
    tokenizer.encode("\n" + phrase),
    tokenizer.encode("\n\n" + phrase.capitalize())
)

[3359] [20790] [198, 28602] [198, 20566] [271, 28602]


In [90]:
_tokens = tokens[0].clone()

for ip, ph in enumerate(phrases[0]):
    positions = action_postions[ph]
    r = mean_reprs_clean[phrases[0][ip]]
    print(
        positions
    )
    for sp, ep in positions:
        _tokens[sp:ep] = 1234
    # _tokens[positions] = 1234

# tokens[0, 25:50]

[(20, 22), (55, 57), (76, 78), (92, 94), (384, 386), (396, 398), (534, 536), (546, 548), (558, 560), (855, 857), (905, 907), (1827, 1829), (1831, 1833), (1870, 1872), (1889, 1891), (1903, 1905), (1928, 1930), (1936, 1938), (1948, 1950), (1997, 1999), (2032, 2034), (2041, 2043), (2078, 2080), (2086, 2088)]
[(31, 34), (114, 117), (131, 134), (154, 157), (379, 382), (503, 506), (516, 519), (529, 532), (859, 862), (947, 951), (2094, 2097), (2099, 2102)]
[(36, 39), (172, 175), (192, 195), (216, 219), (388, 391), (400, 403), (538, 541), (550, 553), (562, 565), (862, 865), (986, 989), (2121, 2124)]
[(24, 26), (237, 239), (260, 262), (279, 281), (371, 374), (495, 498), (508, 511), (521, 524), (857, 859), (1038, 1041), (1317, 1320), (1322, 1324), (1386, 1388), (1404, 1406), (1434, 1436), (1438, 1441), (1449, 1451), (1492, 1494), (1527, 1529), (1550, 1552), (1563, 1565), (1569, 1572), (1606, 1608), (1657, 1659), (1697, 1699), (2148, 2150), (2157, 2159), (2167, 2169)]
[(66, 68), (103, 105), (143,

In [92]:
print(tokenizer.decode(_tokens))

<|im_start|>user
I am playing with a set of objects. Here are the actions I can do

ItemItem object
ItemItem object from another object
ItemItemItem object
ItemItemItem object from another object

I have the following restrictions on my actions:
    ToItemItem action, the following facts need to be trueItemItem objectItemItem objectItemItem.
   ItemItem action is performed the following facts will be trueItemItem object.
   ItemItem action is performed the following facts will be falseItemItem objectItemItem objectItemItem.
    ToItemItemItem action, the following facts need to be trueItemItem object.
   ItemItemItem action is performed the following facts will be trueItemItem objectItemItem objectItemItem.    
   ItemItemItem action is performed the following facts will be falseItemItem object.
    ToItemItemItem action, the following needs to be trueItemItem other objectItemItem object.
   ItemItemItem action is performed the following will be trueItemItemItemItem object,ItemItemItem

In [93]:
from collections import OrderedDict


def forward_hook(module, input, output):
    """Replace output with the mean representation"""
    output = output[0]
    if output.shape[1] == 1:
        return (output,)
    
    output = output[0]    

    print("asdasd")
    
    for ip, ph in enumerate(phrases[0]):
        positions = action_postions[ph]
        r = mean_reprs_clean[phrases[0][ip]] - mean_clean
        for sp, ep in positions:
            output[sp:ep] += torch.tensor(r, dtype=output.dtype, device=output.device)
        # output[positions] = torch.tensor(r, dtype=output.dtype, device=output.device)

    return (output.unsqueeze(0),)
    

for m in model.modules():
    m._forward_hooks = OrderedDict()
    
model.model.layers[47].register_forward_hook(forward_hook)

with torch.no_grad():
    enc = model.generate(tokens.to(device), do_sample=False, max_new_tokens=20000, temperature=None, top_p=None, top_k=None, use_cache=True)


asdasd


In [94]:
print(tokenizer.decode(enc[0, :20000]))

<|im_start|>user
I am playing with a set of objects. Here are the actions I can do

   Attack object
   Feast object from another object
   Succumb object
   Overcome object from another object

I have the following restrictions on my actions:
    To perform Attack action, the following facts need to be true: Province object, Planet object, Harmony.
    Once Attack action is performed the following facts will be true: Pain object.
    Once Attack action is performed the following facts will be false: Province object, Planet object, Harmony.
    To perform Succumb action, the following facts need to be true: Pain object.
    Once Succumb action is performed the following facts will be true: Province object, Planet object, Harmony.    
    Once Succumb action is performed the following facts will be false: Pain object.
    To perform Overcome action, the following needs to be true: Province other object, Pain object.
    Once Overcome action is performed the following will be true: Harmo

In [46]:
enc.shape

torch.Size([1, 16778])

In [53]:
len(tokenize_blocksworld_generation(tokenizer, row)[0])

20191

In [29]:
len(tokenizer.encode(row["generation"]))

23909

In [77]:
print(row["generation"])

Okay, let's see. I need to solve this problem where the initial conditions are given, and I have to come up with a plan using the actions provided to reach the goal. Let me start by understanding the problem step by step.

First, let me restate the initial conditions and the goal to make sure I have them right.

**Initial Conditions:**
- Block A craves Block C (Object Craves other object: A→C)
- Block C craves Block B (C→B)
- Block D craves Block A (D→A)
- Harmony is present (Harmony)
- Planet Block B (Planet B)
- Province Block D (Province D)

**Goal:**
- Block A craves Block D (A→D)
- Block C craves Block A (C→A)
- Block D craves Block B (D→B)

So, I need to manipulate the actions (Attack, Feast, Succumb, Overcome) to transition from the initial state to the goal state. Let me recall the actions and their preconditions and effects.

Let me list out the actions again with their preconditions and effects:

1. **Attack object:**
   - Requires: Province object, Planet object, Harmony.
  